In [382]:
gen_report = False

In [383]:
import os
import json
import pandas as pd

root = "experiments"

records = []

for dirpath, dirnames, filenames in os.walk(root):
    if "report.json" in filenames:
        report_path = os.path.join(dirpath, "report.json")

        with open(report_path, "r") as f:
            data = json.load(f)

        record = {}

        record["VQEL"] = data.get("VQEL")

        # metrics
        for k, v in data.get("metrics", {}).items():
            record[k] = v

        # config
        for k, v in data.get("config", {}).items():
            if 'dir' in k.lower() or 'split' in k.lower():
                continue
            record[k] = v

        record["path"] = dirpath.replace('experiments/', '').replace('/results', '')
        records.append(record)

df = pd.DataFrame(records)
df["sim"] = df["sim"].fillna("euclidean")
df["freeze_object_encoder_a"] = df["freeze_object_encoder_a"].fillna(False)
df["freeze_object_encoder"] = df["freeze_object_encoder"].fillna(False)
df["baseline_a"] = df["baseline_a"].fillna(False)

df.loc[df["learning_rate_phase2_a"].isna(), "learning_rate_phase2_a"] = df["learning_rate_phase2"]
df.loc[df["learning_rate_phase2_b"].isna(), "learning_rate_phase2_b"] = df["learning_rate_phase2"]
df = df.drop(columns=["learning_rate_phase2"])

df.loc[df["self_play_accuracy_a"].isna(), "self_play_accuracy_a"] = df["self_play_accuracy"]
df = df.drop(columns=["self_play_accuracy"])

df.loc[df["pretrained_checkpoint_a"].isna(), "pretrained_checkpoint_a"] = df["pretrained_checkpoint"]
df = df.drop(columns=["pretrained_checkpoint"])


df.loc[df["reset_unfrozen_params"].isna(), "reset_unfrozen_params"] = False

df["reset_unfrozen_params"] = df["reset_unfrozen_params"].map({True: "yes", False: "no"})
df["freeze_codebook"] = df["freeze_codebook"].map({True: "yes", False: "no"})
df["freeze_object_encoder"] = df["freeze_object_encoder"].map({True: "yes", False: "no"})



df = df.where(pd.notna(df), "None")
pd.options.display.float_format = '{:.10g}'.format
df = df.map(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and (abs(x) < 0.01 or abs(x) == 0.1 or abs(x) == 0.01) else x)
print("Total reports loaded:", len(df))


Total reports loaded: 454


/tmp/ipykernel_959836/2341367273.py:35: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["freeze_object_encoder_a"] = df["freeze_object_encoder_a"].fillna(False)
/tmp/ipykernel_959836/2341367273.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["freeze_object_encoder"] = df["freeze_object_encoder"].fillna(False)
/tmp/ipykernel_959836/2341367273.py:37: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future beha

In [384]:
def filter_df(filters, df=df):

    idx = pd.Series(True, index=df.index)

    for key, values in filters.items():
        mask = pd.Series(False, index=df.index)
        if not isinstance(values, (list, tuple, set)):
            values = [values]
        for value in values:
            if value == "!None":
                mask |= (df[key] != "None")
            else:
                mask |= (df[key] == value)
        idx &= mask 

    return df[idx].sort_values(by=["seed", "freeze_object_encoder", "reset_unfrozen_params", "freeze_codebook", "agent_a_training_mode", "freeze_agent_b", "sampling_temperature", "commitment_weight", "entropy_regularization_factor", "learning_rate_phase1", "learning_rate_phase2_b", "learning_rate_phase2_a"])
    

In [385]:
import base64

def to_html(df):    

    # ---- highlight rule ----
    if 'mutual_play_accuracy' in df:
        max_col = 'mutual_play_accuracy'
    else:
        max_col = 'test_accuracy'
    max_val = pd.to_numeric(df[max_col]).max()

    def highlight_max_row(row):
        if  pd.to_numeric(row[max_col]) == max_val:
            return ['font-weight: bold; background-color: #ffff99'] * len(row)
        else:
            return [''] * len(row)

    # ---- style ----
    styled = (
        df.style
            .apply(highlight_max_row, axis=1)  # <<< APPLY HIGHLIGHT HERE
            .hide(axis="index")
            .format(lambda x: f"{x:.0e}" if isinstance(x, (int, float)) and x != 0 and abs(x) < 0.01 else x)
            .set_table_styles([
                {"selector": "td", "props": [("border-right", "1px solid black")]},
                {"selector": "th", "props": [
                    ("border-right", "1px solid black"),
                    ("color", "darkblue"),
                    ("font-weight", "bold"),
                    ("padding-left", "8px"),
                    ("padding-right", "8px")
                ]},
            ])
            .set_properties(**{"text-align": "center"})
    )

    html_table = styled.to_html(index=False)

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_table)

        
def add_heading(num=1, title=""):
    """Add a professional-colored heading to results.html"""
    colors = ["#0b3d91",  # dark blue
              "#800000",  # maroon
              "#205522",  # dark green
              "#4b0082",  # indigo
              "#444444"]  # dark gray
    color = colors[(num-1) % len(colors)]  # cycle through colors
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f'<h{num} style="color:{color}; font-family:Arial, sans-serif;">{title}</h{num}>\n')
        
def clear():
    if gen_report:
        with open("results.html", "w") as f:
            f.write("")

def line():
    if gen_report:
        with open("results.html", "a") as f:
            f.write('<hr style="border:1px solid #444; margin:10px 0;">\n')
        
        
def write(text):
    html_text = text.replace("\n", "<br>\n")
    
    if gen_report:
        with open("results.html", "a") as f:
            f.write(f"{html_text}\n")


def add_plot(path="plot.png"):
    """Embed an image file directly into results.html using base64."""
    with open(path, "rb") as img:
        encoded = base64.b64encode(img.read()).decode("utf-8")

    html_img = (
        '<img src="data:image/png;base64,' +
        encoded +
        '" style="max-width:650px; height:auto;"><br>\n'
    )

    if gen_report:
        with open("results.html", "a") as f:
            f.write(html_img)
        
    if os.path.exists(path):
        os.remove(path)

In [386]:
import os
import shutil

def remove_expr(res_df):
    for path in res_df["path"]:
        full_path = os.path.join("experiments", path)

        if not os.path.isdir(full_path):
            print(f"[SKIP] Not found: {full_path}")
            continue

        answer = input(f"⚠️ Delete expr '{path}' ? (y/n): ").strip().lower()

        if answer == "y":
            shutil.rmtree(full_path)
            print(f"✅ Deleted: {path}")
        else:
            print(f"❌ Skipped: {path}")

In [387]:
from itertools import product
import pandas as pd


def extract_maxes(df, cols=["seed", "agent_a_training_mode"], max_col="mutual_play_accuracy"):
    values = []
    for col in cols:
        value = set(df[col])
        values.append(value)

    combinations = [list(x) for x in product(*values)]

    maxes = []
    for comb in combinations:
        section = filter_df({col: value for col, value in zip(cols, comb)}, df)
        if not section.empty:
            max_row = section.loc[section["mutual_play_accuracy"].idxmax()]
            maxes.append(max_row.to_frame().T)

    final_df = pd.concat(maxes, ignore_index=True)
    return final_df.sort_values(by=["agent_a_training_mode"])


def mean_and_std(df, out_cols=["VQEL", "dataset", "sim", "agent_a_training_mode"]):
    rows = []

    for i in range(0, len(df), 3):
        chunk = df.iloc[i:i+3]

        mean = chunk["mutual_play_accuracy"].mean() * 100
        std = chunk["mutual_play_accuracy"].std() * 100

        rows.append({
            col: chunk[col].iloc[0] for col in out_cols} | {
            "mutual_play_accuracy": f"{mean:.1f} ± {std:.1f}"
        })

    out = pd.DataFrame(rows)
    return out.sort_values(by=out_cols)
    

In [388]:
clear()

---

# EXP1: VQEL (3 Modes) vs. Baseline

In [389]:
add_heading(1, "EXP1: VQEL (3 Modes) vs. VQ-Reinforce vs. Baseline")

write(
"""
<b>VQEL - Euclidean or Cosine</b>
Here, we train VQEL in the self-play phase for 50 epochs, followed by another 50 epochs in mutual play.
Mutual play has three modes that determine the training behavior of the sender:

- Frozen: the sender's parameters are fixed.
- Reinforce only: the sender is fine tuned using the reinforce algorithm.
- Reinforce with preservation: the sender is fine tuned using both the reinforce loss and the self-play loss.

We also use two different codebooks, depending on the distance metric used to retrieve the nearest vector:
- Euclidean distance
- Cosine similarity

<b>VQ-Reinforce</b>
The architecture of VQ-Reinforce is the same as VQEL, but it is trained for 100 epochs using REINFORCE, without any self-play.

<b>Baseline</b>
The baseline is trained for 100 epochs.

"""
)

In [390]:
vq_cols = [
    "seed",
    "VQEL",
    "dataset",
    "sim",
    "agent_a_training_mode", 
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
    "commitment_weight",
    "sampling_temperature",
    "representation_dim",
    "pretrained_checkpoint_a",
    "path",
]

vq_rf_cols = [
    "seed",
    "VQEL",
    "dataset",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "mutual_play_accuracy",
    "commitment_weight",
    "sampling_temperature",
    "representation_dim",
    "agent_a_training_mode",
    "num_pretrain_epochs",
    "contrastive_loss_temperature",
    "path"
]

bs_cols = [
    "VQEL",
    "dataset",
    "contrastive_loss_temperature",
    "entropy_regularization_factor",
    "learning_rate",
    "test_accuracy",
    "sampling_temperature",
    "representation_dim",
    "path"
]

vq_rf_cols_report = [
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "commitment_weight",
    "sampling_temperature",
    "mutual_play_accuracy",
]

vq_cols_report = [
    "agent_a_training_mode",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "mutual_play_accuracy",
]

bs_cols_report = [
    "learning_rate",
    "test_accuracy",
]

## Objects

### VQEL - Euclidean

In [391]:
add_heading(2, "Objects")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "objects",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
250,1,True,objects,euclidean,frozen,1e-03,1e-03,1e-03,0.721,0.786,0.25,1e-05,32,None,20251118_1104_bs32_vocab10_repr32_msglen4_lr1_...
430,1,True,objects,euclidean,frozen,1e-04,1e-04,1e-04,0.393,0.455,0.25,1e-05,32,None,20251118_1116_bs32_vocab10_repr32_msglen4_lr1_...
308,1,True,objects,euclidean,frozen,1e-05,1e-05,1e-05,0.155,0.179,0.25,1e-05,32,None,20251118_1129_bs32_vocab10_repr32_msglen4_lr1_...
232,1,True,objects,euclidean,reinforce_only,1e-03,1e-03,1e-03,0.721,0.282,0.25,1e-05,32,20251118_1104_bs32_vocab10_repr32_msglen4_lr1_...,20251207_0950_bs32_vocab10_repr32_msglen4_lr1_...
122,1,True,objects,euclidean,reinforce_only,1e-03,1e-04,1e-03,0.721,0.786,0.25,1e-05,32,20251118_1104_bs32_vocab10_repr32_msglen4_lr1_...,20251211_0115_bs32_vocab10_repr32_msglen4_lr1_...
17,1,True,objects,euclidean,reinforce_only,1e-03,1e-05,1e-03,0.721,0.841,0.25,1e-05,32,20251118_1104_bs32_vocab10_repr32_msglen4_lr1_...,20251211_0124_bs32_vocab10_repr32_msglen4_lr1_...
446,1,True,objects,euclidean,reinforce_only,1e-03,1e-06,1e-03,0.721,0.819,0.25,1e-05,32,20251118_1104_bs32_vocab10_repr32_msglen4_lr1_...,20251211_0133_bs32_vocab10_repr32_msglen4_lr1_...
245,1,True,objects,euclidean,reinforce_only,1e-03,1e-04,1e-04,0.721,0.552,0.25,1e-05,32,20251118_1104_bs32_vocab10_repr32_msglen4_lr1_...,20251207_0958_bs32_vocab10_repr32_msglen4_lr1_...
185,1,True,objects,euclidean,reinforce_only,1e-03,1e-05,1e-05,0.721,0.19,0.25,1e-05,32,20251118_1104_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1006_bs32_vocab10_repr32_msglen4_lr1_...
411,1,True,objects,euclidean,reinforce_with_preservation,1e-03,1e-03,1e-03,0.721,0.605,0.25,1e-05,32,20251118_1104_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1015_bs32_vocab10_repr32_msglen4_lr1_...


### VQEL - Cosine

In [392]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "objects",
    "sim": "cosine",
    "VQEL": True,
    "num_pretrain_epochs": 50,
    "pretrained_checkpoint_b": "None",
})

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
93,1,True,objects,cosine,frozen,1e-03,1e-03,1e-03,0.826,0.847,0.25,1e-05,32,None,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...
425,1,True,objects,cosine,frozen,1e-03,1e-04,1e-04,0.826,0.666,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251205_2054_bs32_vocab10_repr32_msglen4_lr1_...
352,1,True,objects,cosine,frozen,1e-03,1e-05,1e-05,0.826,0.25,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251205_2059_bs32_vocab10_repr32_msglen4_lr1_...
111,1,True,objects,cosine,frozen,1e-04,1e-04,1e-04,0.635,0.618,0.25,1e-05,32,None,20251129_2058_bs32_vocab10_repr32_msglen4_lr1_...
31,1,True,objects,cosine,frozen,1e-05,1e-05,1e-05,0.086,0.123,0.25,1e-05,32,None,20251205_1712_bs32_vocab10_repr32_msglen4_lr1_...
261,1,True,objects,cosine,reinforce_only,1e-03,1e-03,1e-03,0.826,0.714,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251205_1943_bs32_vocab10_repr32_msglen4_lr1_...
438,1,True,objects,cosine,reinforce_only,1e-03,1e-04,1e-03,0.826,0.831,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251209_2347_bs32_vocab10_repr32_msglen4_lr1_...
200,1,True,objects,cosine,reinforce_only,1e-03,1e-05,1e-03,0.826,0.849,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251209_2355_bs32_vocab10_repr32_msglen4_lr1_...
188,1,True,objects,cosine,reinforce_only,1e-03,1e-06,1e-03,0.826,0.85,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251210_2243_bs32_vocab10_repr32_msglen4_lr1_...
293,1,True,objects,cosine,reinforce_only,1e-03,1e-04,1e-04,0.826,0.613,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251205_1952_bs32_vocab10_repr32_msglen4_lr1_...


In [ ]:
final = extract_maxes(res)

final[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
0,1,True,objects,cosine,frozen,1e-03,1e-03,1e-03,0.826,0.847,0.25,1e-05,32,None,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...
3,2,True,objects,cosine,frozen,1e-03,1e-03,1e-03,0.826,0.839,0.25,1e-05,32,None,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...
6,3,True,objects,cosine,frozen,1e-03,1e-03,1e-03,0.817,0.856,0.25,1e-05,32,None,20251216_1609_bs32_vocab10_repr32_msglen4_lr1_...
2,1,True,objects,cosine,reinforce_only,1e-03,1e-06,1e-03,0.826,0.85,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251210_2243_bs32_vocab10_repr32_msglen4_lr1_...
5,2,True,objects,cosine,reinforce_only,1e-03,1e-05,1e-03,0.826,0.855,0.25,1e-05,32,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251217_0000_bs32_vocab10_repr32_msglen4_lr1_...
8,3,True,objects,cosine,reinforce_only,1e-03,1e-06,1e-03,0.817,0.867,0.25,1e-05,32,20251216_1609_bs32_vocab10_repr32_msglen4_lr1_...,20251217_0032_bs32_vocab10_repr32_msglen4_lr1_...
1,1,True,objects,cosine,reinforce_with_preservation,1e-03,1e-06,1e-03,0.826,0.855,0.25,1e-05,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251210_1028_bs32_vocab10_repr32_msglen4_lr1_...
4,2,True,objects,cosine,reinforce_with_preservation,1e-03,1e-05,1e-03,0.826,0.862,0.25,1e-05,32,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251217_0050_bs32_vocab10_repr32_msglen4_lr1_...
7,3,True,objects,cosine,reinforce_with_preservation,1e-03,1e-06,1e-03,0.817,0.874,0.25,1e-05,32,20251216_1609_bs32_vocab10_repr32_msglen4_lr1_...,20251217_0125_bs32_vocab10_repr32_msglen4_lr1_...


In [394]:
mean_and_std(final)

,VQEL,dataset,sim,agent_a_training_mode,mutual_play_accuracy
0,True,objects,cosine,frozen,84.7 ± 0.9
1,True,objects,cosine,reinforce_only,85.7 ± 0.9
2,True,objects,cosine,reinforce_with_preservation,86.4 ± 1.0


### VQ-Reinforce

In [395]:
add_heading(3, "VQ-Reinforce")

res = filter_df({
    "dataset": "objects",
    "VQEL": True,
    "num_pretrain_epochs": 0
})

to_html(res[vq_rf_cols_report])

res[vq_rf_cols]

,seed,VQEL,dataset,learning_rate_phase2_a,learning_rate_phase2_b,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,agent_a_training_mode,num_pretrain_epochs,contrastive_loss_temperature,path
2,1,True,objects,1e-03,1e-03,0.071,0,1e-02,32,reinforce_only,0,1e-02,20251209_2229_bs32_vocab10_repr32_msglen4_lr1_...
146,1,True,objects,1e-04,1e-04,0.293,0.25,1e-02,32,reinforce_only,0,1e-02,20251206_0704_bs32_vocab10_repr32_msglen4_lr1_...
310,1,True,objects,1e-05,1e-05,0.052,0.25,1e-02,32,reinforce_only,0,1e-02,20251206_0722_bs32_vocab10_repr32_msglen4_lr1_...


### Baseline

In [396]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "objects",
    "VQEL": False,
})

to_html(res[bs_cols_report])

res[bs_cols]

,VQEL,dataset,contrastive_loss_temperature,entropy_regularization_factor,learning_rate,test_accuracy,sampling_temperature,representation_dim,path
140,False,objects,1e-02,0,1e-04,0.017,1,32,20251120_1111_bs32_vocab10_repr32_msglen4_lr0....
202,False,objects,1e-02,0,1e-05,0.014,1,32,20251120_1121_bs32_vocab10_repr32_msglen4_lr1e...
410,False,objects,1e-02,0,1e-03,0.592,1,32,20251120_1100_bs32_vocab10_repr32_msglen4_lr0....


## ShapeWorld

### VQEL - Euclidean

In [397]:
add_heading(2, "ShapeWorld")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "shape",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
322,1,True,shape,euclidean,frozen,1e-03,1e-03,1e-03,0.755,0.761,0.25,1e-05,1024,None,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...
94,1,True,shape,euclidean,frozen,1e-04,1e-04,1e-04,0.696,0.696,0.25,1e-05,1024,None,20251120_1950_bs32_vocab10_repr1024_msglen4_lr...
39,1,True,shape,euclidean,frozen,1e-05,1e-05,1e-05,0.395,0.454,0.25,1e-05,1024,None,20251120_2033_bs32_vocab10_repr1024_msglen4_lr...
300,1,True,shape,euclidean,reinforce_only,1e-03,1e-03,1e-03,0.755,0.279,0.25,1e-05,1024,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...,20251207_0523_bs32_vocab10_repr1024_msglen4_lr...
272,1,True,shape,euclidean,reinforce_only,1e-03,1e-04,1e-03,0.755,0.751,0.25,1e-05,1024,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...,20251211_0210_bs32_vocab10_repr1024_msglen4_lr...
96,1,True,shape,euclidean,reinforce_only,1e-03,1e-05,1e-03,0.755,0.838,0.25,1e-05,1024,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...,20251211_0231_bs32_vocab10_repr1024_msglen4_lr...
155,1,True,shape,euclidean,reinforce_only,1e-03,1e-06,1e-03,0.755,0.835,0.25,1e-05,1024,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...,20251211_0252_bs32_vocab10_repr1024_msglen4_lr...
35,1,True,shape,euclidean,reinforce_only,1e-03,1e-04,1e-04,0.755,0.744,0.25,1e-05,1024,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...,20251207_0542_bs32_vocab10_repr1024_msglen4_lr...
66,1,True,shape,euclidean,reinforce_only,1e-03,1e-05,1e-05,0.755,0.845,0.25,1e-05,1024,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...,20251207_0602_bs32_vocab10_repr1024_msglen4_lr...
236,1,True,shape,euclidean,reinforce_with_preservation,1e-03,1e-03,1e-03,0.755,0.294,0.25,1e-05,1024,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...,20251207_0622_bs32_vocab10_repr1024_msglen4_lr...


### VQEL - Cosine

In [398]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "num_pretrain_epochs": 50,
    "pretrained_checkpoint_b": "None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
289,1,True,shape,cosine,frozen,1e-03,1e-03,1e-03,0.863,0.887,0.25,1e-05,1024,None,20251125_0021_bs32_vocab10_repr1024_msglen4_lr...
60,1,True,shape,cosine,frozen,1e-04,1e-03,1e-03,0.869,0.852,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251205_2126_bs32_vocab10_repr1024_msglen4_lr...
108,1,True,shape,cosine,frozen,1e-04,1e-04,1e-04,0.869,0.881,0.25,1e-05,1024,None,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...
137,1,True,shape,cosine,frozen,1e-04,1e-05,1e-05,0.869,0.895,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251205_2139_bs32_vocab10_repr1024_msglen4_lr...
225,1,True,shape,cosine,frozen,1e-05,1e-05,1e-05,0.764,0.869,0.25,1e-05,1024,None,20251125_0126_bs32_vocab10_repr1024_msglen4_lr...
254,1,True,shape,cosine,reinforce_only,1e-04,1e-05,1e-03,0.869,0.847,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251210_1047_bs32_vocab10_repr1024_msglen4_lr...
174,1,True,shape,cosine,reinforce_only,1e-04,1e-04,1e-04,0.869,0.763,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251205_2153_bs32_vocab10_repr1024_msglen4_lr...
179,1,True,shape,cosine,reinforce_only,1e-04,1e-05,1e-04,0.869,0.88,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251210_0043_bs32_vocab10_repr1024_msglen4_lr...
117,1,True,shape,cosine,reinforce_only,1e-04,1e-06,1e-04,0.869,0.9,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251210_0023_bs32_vocab10_repr1024_msglen4_lr...
167,1,True,shape,cosine,reinforce_only,1e-04,1e-07,1e-04,0.869,0.897,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251210_1108_bs32_vocab10_repr1024_msglen4_lr...


In [399]:
final = extract_maxes(res)
final[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
0,1,True,shape,cosine,frozen,1e-04,1e-05,1e-05,0.869,0.895,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251205_2139_bs32_vocab10_repr1024_msglen4_lr...
3,2,True,shape,cosine,frozen,1e-03,1e-03,1e-03,0.888,0.884,0.25,1e-05,1024,None,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...
6,3,True,shape,cosine,frozen,1e-03,1e-03,1e-03,0.863,0.888,0.25,1e-05,1024,None,20251216_1649_bs32_vocab10_repr1024_msglen4_lr...
2,1,True,shape,cosine,reinforce_only,1e-04,1e-06,1e-04,0.869,0.9,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251210_0023_bs32_vocab10_repr1024_msglen4_lr...
5,2,True,shape,cosine,reinforce_only,1e-03,1e-06,1e-04,0.888,0.913,0.25,1e-05,1024,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251217_0154_bs32_vocab10_repr1024_msglen4_lr...
8,3,True,shape,cosine,reinforce_only,1e-03,1e-05,1e-04,0.863,0.919,0.25,1e-05,1024,20251216_1649_bs32_vocab10_repr1024_msglen4_lr...,20251217_0254_bs32_vocab10_repr1024_msglen4_lr...
1,1,True,shape,cosine,reinforce_with_preservation,1e-04,1e-07,1e-05,0.869,0.894,0.25,1e-05,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251211_0001_bs32_vocab10_repr1024_msglen4_lr...
4,2,True,shape,cosine,reinforce_with_preservation,1e-03,1e-06,1e-04,0.888,0.909,0.25,1e-05,1024,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251217_0438_bs32_vocab10_repr1024_msglen4_lr...
7,3,True,shape,cosine,reinforce_with_preservation,1e-03,1e-05,1e-04,0.863,0.924,0.25,1e-05,1024,20251216_1649_bs32_vocab10_repr1024_msglen4_lr...,20251217_0638_bs32_vocab10_repr1024_msglen4_lr...


In [400]:
mean_and_std(final)

,VQEL,dataset,sim,agent_a_training_mode,mutual_play_accuracy
0,True,shape,cosine,frozen,88.9 ± 0.6
1,True,shape,cosine,reinforce_only,91.1 ± 1.0
2,True,shape,cosine,reinforce_with_preservation,90.9 ± 1.5


### VQ-Reinforce

In [401]:
add_heading(3, "VQ-Reinforce")

res = filter_df({
    "dataset": "shape",
    "VQEL": True,
    "num_pretrain_epochs": 0
})

to_html(res[vq_rf_cols_report])

res[vq_rf_cols]

,seed,VQEL,dataset,learning_rate_phase2_a,learning_rate_phase2_b,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,agent_a_training_mode,num_pretrain_epochs,contrastive_loss_temperature,path
182,1,True,shape,1e-03,1e-03,0.753,0.25,1e-01,1024,reinforce_only,0,1e-02,20251206_0950_bs32_vocab10_repr1024_msglen4_lr...
84,1,True,shape,1e-04,1e-04,0.862,0.25,1e-01,1024,reinforce_only,0,1e-02,20251206_1032_bs32_vocab10_repr1024_msglen4_lr...
214,1,True,shape,1e-05,1e-05,0.865,0.25,1e-01,1024,reinforce_only,0,1e-02,20251206_1114_bs32_vocab10_repr1024_msglen4_lr...


### Baseline

In [402]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "shape",
    "VQEL": False,
    "representation_dim": 1024,
    "seed": 1
})

to_html(res[bs_cols_report])

res[bs_cols]

,VQEL,dataset,contrastive_loss_temperature,entropy_regularization_factor,learning_rate,test_accuracy,sampling_temperature,representation_dim,path
281,False,shape,1e-02,0,1e-03,0.307,1,1024,20251120_1908_bs32_vocab10_repr1024_msglen4_lr...
316,False,shape,1e-02,0,1e-04,0.86,1,1024,20251120_1947_bs32_vocab10_repr1024_msglen4_lr...
373,False,shape,1e-02,0,1e-05,0.857,1,1024,20251120_2027_bs32_vocab10_repr1024_msglen4_lr...


## DSprites

### VQEL - Euclidean

In [403]:
add_heading(2, "DSprites")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "dsprites",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
356,1,True,dsprites,euclidean,frozen,1e-03,1e-03,1e-03,0.773,0.732,0.25,1e-05,1024,None,20251122_2052_bs32_vocab10_repr1024_msglen4_lr...
382,1,True,dsprites,euclidean,frozen,1e-04,1e-04,1e-04,0.744,0.662,0.25,1e-05,1024,None,20251122_2146_bs32_vocab10_repr1024_msglen4_lr...
132,1,True,dsprites,euclidean,frozen,1e-05,1e-05,1e-05,0.423,0.491,0.25,1e-05,1024,None,20251122_2224_bs32_vocab10_repr1024_msglen4_lr...
257,1,True,dsprites,euclidean,reinforce_only,1e-03,1e-03,1e-03,0.773,0.304,0.25,1e-05,1024,20251122_2052_bs32_vocab10_repr1024_msglen4_lr...,20251207_0735_bs32_vocab10_repr1024_msglen4_lr...
164,1,True,dsprites,euclidean,reinforce_only,1e-03,1e-04,1e-03,0.773,0.701,0.25,1e-05,1024,20251122_2052_bs32_vocab10_repr1024_msglen4_lr...,20251211_0425_bs32_vocab10_repr1024_msglen4_lr...
149,1,True,dsprites,euclidean,reinforce_only,1e-03,1e-05,1e-03,0.773,0.865,0.25,1e-05,1024,20251122_2052_bs32_vocab10_repr1024_msglen4_lr...,20251211_0446_bs32_vocab10_repr1024_msglen4_lr...
51,1,True,dsprites,euclidean,reinforce_only,1e-03,1e-06,1e-03,0.773,0.873,0.25,1e-05,1024,20251122_2052_bs32_vocab10_repr1024_msglen4_lr...,20251211_0506_bs32_vocab10_repr1024_msglen4_lr...
76,1,True,dsprites,euclidean,reinforce_only,1e-03,1e-04,1e-04,0.773,0.777,0.25,1e-05,1024,20251122_2052_bs32_vocab10_repr1024_msglen4_lr...,20251207_0755_bs32_vocab10_repr1024_msglen4_lr...
378,1,True,dsprites,euclidean,reinforce_only,1e-03,1e-05,1e-05,0.773,0.863,0.25,1e-05,1024,20251122_2052_bs32_vocab10_repr1024_msglen4_lr...,20251207_0815_bs32_vocab10_repr1024_msglen4_lr...
170,1,True,dsprites,euclidean,reinforce_with_preservation,1e-03,1e-03,1e-03,0.773,0.568,0.25,1e-05,1024,20251122_2052_bs32_vocab10_repr1024_msglen4_lr...,20251207_0835_bs32_vocab10_repr1024_msglen4_lr...


### VQEL - Cosine

In [404]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "dsprites",
    "sim": "cosine",
    "VQEL": True,
    "num_pretrain_epochs": 50,
    "pretrained_checkpoint_b": "None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
44,1,True,dsprites,cosine,frozen,1e-03,1e-03,1e-03,0.883,0.895,0.25,1e-05,1024,None,20251124_2057_bs32_vocab10_repr1024_msglen4_lr...
338,1,True,dsprites,cosine,frozen,1e-04,1e-03,1e-03,0.929,0.91,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251205_2348_bs32_vocab10_repr1024_msglen4_lr...
238,1,True,dsprites,cosine,frozen,1e-04,1e-04,1e-04,0.929,0.931,0.25,1e-05,1024,None,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...
9,1,True,dsprites,cosine,frozen,1e-04,1e-05,1e-05,0.929,0.924,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251206_0002_bs32_vocab10_repr1024_msglen4_lr...
73,1,True,dsprites,cosine,frozen,1e-05,1e-05,1e-05,0.747,0.812,0.25,1e-05,1024,None,20251124_2213_bs32_vocab10_repr1024_msglen4_lr...
434,1,True,dsprites,cosine,reinforce_only,1e-04,1e-05,1e-03,0.929,0.874,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251210_1242_bs32_vocab10_repr1024_msglen4_lr...
133,1,True,dsprites,cosine,reinforce_only,1e-04,1e-04,1e-04,0.929,0.769,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251206_0015_bs32_vocab10_repr1024_msglen4_lr...
368,1,True,dsprites,cosine,reinforce_only,1e-04,1e-05,1e-04,0.929,0.917,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251210_0215_bs32_vocab10_repr1024_msglen4_lr...
375,1,True,dsprites,cosine,reinforce_only,1e-04,1e-06,1e-04,0.929,0.942,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251210_0154_bs32_vocab10_repr1024_msglen4_lr...
384,1,True,dsprites,cosine,reinforce_only,1e-04,1e-07,1e-04,0.929,0.923,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251210_1221_bs32_vocab10_repr1024_msglen4_lr...


In [405]:
final = extract_maxes(res)
final[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
0,1,True,dsprites,cosine,frozen,1e-04,1e-04,1e-04,0.929,0.931,0.25,1e-05,1024,None,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...
3,2,True,dsprites,cosine,frozen,1e-04,1e-04,1e-04,0.899,0.918,0.25,1e-05,1024,None,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...
6,3,True,dsprites,cosine,frozen,1e-04,1e-04,1e-04,0.906,0.916,0.25,1e-05,1024,None,20251216_1900_bs32_vocab10_repr1024_msglen4_lr...
2,1,True,dsprites,cosine,reinforce_only,1e-04,1e-06,1e-04,0.929,0.942,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251210_0154_bs32_vocab10_repr1024_msglen4_lr...
5,2,True,dsprites,cosine,reinforce_only,1e-04,1e-06,1e-04,0.899,0.92,0.25,1e-05,1024,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251217_0923_bs32_vocab10_repr1024_msglen4_lr...
8,3,True,dsprites,cosine,reinforce_only,1e-04,1e-06,1e-04,0.906,0.916,0.25,1e-05,1024,20251216_1900_bs32_vocab10_repr1024_msglen4_lr...,20251217_1213_bs32_vocab10_repr1024_msglen4_lr...
1,1,True,dsprites,cosine,reinforce_with_preservation,1e-04,1e-06,1e-05,0.929,0.934,0.25,1e-05,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251211_0049_bs32_vocab10_repr1024_msglen4_lr...
4,2,True,dsprites,cosine,reinforce_with_preservation,1e-04,1e-06,1e-04,0.899,0.922,0.25,1e-05,1024,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251217_1340_bs32_vocab10_repr1024_msglen4_lr...
7,3,True,dsprites,cosine,reinforce_with_preservation,1e-04,1e-06,1e-04,0.906,0.919,0.25,1e-05,1024,20251216_1900_bs32_vocab10_repr1024_msglen4_lr...,20251217_1518_bs32_vocab10_repr1024_msglen4_lr...


In [406]:
mean_and_std(final)

,VQEL,dataset,sim,agent_a_training_mode,mutual_play_accuracy
0,True,dsprites,cosine,frozen,92.2 ± 0.8
1,True,dsprites,cosine,reinforce_only,92.6 ± 1.4
2,True,dsprites,cosine,reinforce_with_preservation,92.5 ± 0.8


### VQ-Reinforce

In [407]:
add_heading(3, "VQ-Reinforce")

res = filter_df({
    "dataset": "dsprites",
    "VQEL": True,
    "num_pretrain_epochs": 0
})

to_html(res[vq_rf_cols_report])

res[vq_rf_cols]

,seed,VQEL,dataset,learning_rate_phase2_a,learning_rate_phase2_b,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,agent_a_training_mode,num_pretrain_epochs,contrastive_loss_temperature,path
115,1,True,dsprites,1e-03,1e-03,0.743,0.25,1e-01,1024,reinforce_only,0,1e-02,20251206_0740_bs32_vocab10_repr1024_msglen4_lr...
52,1,True,dsprites,1e-04,1e-04,0.901,0.25,1e-01,1024,reinforce_only,0,1e-02,20251206_0823_bs32_vocab10_repr1024_msglen4_lr...
449,1,True,dsprites,1e-05,1e-05,0.862,0.25,1e-01,1024,reinforce_only,0,1e-02,20251206_0905_bs32_vocab10_repr1024_msglen4_lr...


### Baseline

In [408]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "dsprites",
    "VQEL": False,
})

to_html(res[bs_cols_report])

res[bs_cols]

,VQEL,dataset,contrastive_loss_temperature,entropy_regularization_factor,learning_rate,test_accuracy,sampling_temperature,representation_dim,path
22,False,dsprites,1e-02,0,1e-05,0.86,1,1024,20251123_1207_bs32_vocab10_repr1024_msglen4_lr...
309,False,dsprites,1e-02,0,1e-03,0.069,1,1024,20251123_1103_bs32_vocab10_repr1024_msglen4_lr...
444,False,dsprites,1e-02,0,1e-04,0.857,1,1024,20251123_1135_bs32_vocab10_repr1024_msglen4_lr...


## CelebA

### VQEL - Euclidean

In [409]:
add_heading(2, "CelebA")
add_heading(3, "VQEL - Euclidean")

res = filter_df({
    "dataset": "celeba",
    "sim": "euclidean",
    "VQEL": True,
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
244,1,True,celeba,euclidean,frozen,1e-03,1e-03,1e-03,0.428,0.536,0.25,1e-05,384,None,20251120_1012_bs32_vocab10_repr384_msglen4_lr1...
297,1,True,celeba,euclidean,frozen,1e-04,1e-04,1e-04,0.594,0.604,0.25,1e-05,384,None,20251120_1018_bs32_vocab10_repr384_msglen4_lr1...
0,1,True,celeba,euclidean,frozen,1e-05,1e-05,1e-05,0.4,0.428,0.25,1e-05,384,None,20251120_1025_bs32_vocab10_repr384_msglen4_lr1...
264,1,True,celeba,euclidean,reinforce_only,1e-04,1e-04,1e-04,0.074,0.151,0.25,1e-05,384,20251120_1018_bs32_vocab10_repr384_msglen4_lr1...,20251211_0639_bs32_vocab10_repr384_msglen4_lr1...
364,1,True,celeba,euclidean,reinforce_only,1e-04,1e-05,1e-04,0.074,0.598,0.25,1e-05,384,20251120_1018_bs32_vocab10_repr384_msglen4_lr1...,20251211_0718_bs32_vocab10_repr384_msglen4_lr1...
275,1,True,celeba,euclidean,reinforce_only,1e-04,1e-06,1e-04,0.074,0.52,0.25,1e-05,384,20251120_1018_bs32_vocab10_repr384_msglen4_lr1...,20251211_0757_bs32_vocab10_repr384_msglen4_lr1...
362,1,True,celeba,euclidean,reinforce_only,1e-04,1e-07,1e-04,0.074,0.447,0.25,1e-05,384,20251120_1018_bs32_vocab10_repr384_msglen4_lr1...,20251211_0836_bs32_vocab10_repr384_msglen4_lr1...
299,1,True,celeba,euclidean,reinforce_with_preservation,1e-04,1e-04,1e-04,0.074,0.269,0.25,1e-05,384,20251120_1018_bs32_vocab10_repr384_msglen4_lr1...,20251211_0915_bs32_vocab10_repr384_msglen4_lr1...
176,1,True,celeba,euclidean,reinforce_with_preservation,1e-04,1e-05,1e-04,0.074,0.578,0.25,1e-05,384,20251120_1018_bs32_vocab10_repr384_msglen4_lr1...,20251211_0958_bs32_vocab10_repr384_msglen4_lr1...
201,1,True,celeba,euclidean,reinforce_with_preservation,1e-04,1e-06,1e-04,0.074,0.553,0.25,1e-05,384,20251120_1018_bs32_vocab10_repr384_msglen4_lr1...,20251211_1042_bs32_vocab10_repr384_msglen4_lr1...


### VQEL - Cosine

In [ ]:
add_heading(3, "VQEL - Cosine")

res = filter_df({
    "dataset": "celeba",
    "sim": "cosine",
    "VQEL": True,
    "num_pretrain_epochs": 50,
    "pretrained_checkpoint_b": "None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
120,1,True,celeba,cosine,frozen,1e-03,1e-03,1e-03,0.897,0.907,0.25,1e-05,384,None,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...
423,1,True,celeba,cosine,frozen,1e-03,1e-04,1e-04,0.897,0.912,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251206_0146_bs32_vocab10_repr384_msglen4_lr1...
256,1,True,celeba,cosine,frozen,1e-03,1e-05,1e-05,0.897,0.908,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251206_0208_bs32_vocab10_repr384_msglen4_lr1...
327,1,True,celeba,cosine,frozen,1e-04,1e-04,1e-04,0.885,0.907,0.25,1e-05,384,None,20251201_1559_bs32_vocab10_repr384_msglen4_lr1...
317,1,True,celeba,cosine,frozen,1e-05,1e-05,1e-05,0.716,0.819,0.25,1e-05,384,None,20251201_1700_bs32_vocab10_repr384_msglen4_lr1...
450,1,True,celeba,cosine,reinforce_only,1e-03,1e-03,1e-03,0.897,0.253,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251206_0230_bs32_vocab10_repr384_msglen4_lr1...
416,1,True,celeba,cosine,reinforce_only,1e-03,1e-04,1e-03,0.897,0.875,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251210_0447_bs32_vocab10_repr384_msglen4_lr1...
215,1,True,celeba,cosine,reinforce_only,1e-03,1e-05,1e-03,0.897,0.901,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251210_0528_bs32_vocab10_repr384_msglen4_lr1...
4,1,True,celeba,cosine,reinforce_only,1e-03,1e-06,1e-03,0.897,0.903,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251210_1352_bs32_vocab10_repr384_msglen4_lr1...
286,1,True,celeba,cosine,reinforce_only,1e-03,1e-04,1e-04,0.897,0.897,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251206_0310_bs32_vocab10_repr384_msglen4_lr1...


In [411]:
final = extract_maxes(res)
final[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,pretrained_checkpoint_a,path
0,1,True,celeba,cosine,frozen,1e-03,1e-04,1e-04,0.897,0.912,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251206_0146_bs32_vocab10_repr384_msglen4_lr1...
3,2,True,celeba,cosine,frozen,1e-03,1e-04,1e-04,0.884,0.896,0.25,1e-05,384,None,20251214_2233_bs32_vocab10_repr384_msglen4_lr1...
6,3,True,celeba,cosine,frozen,1e-04,1e-04,1e-04,0.885,0.903,0.25,1e-05,384,None,20251218_0228_bs32_vocab10_repr384_msglen4_lr1...
2,1,True,celeba,cosine,reinforce_only,1e-03,1e-06,1e-04,0.897,0.916,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251210_0326_bs32_vocab10_repr384_msglen4_lr1...
5,2,True,celeba,cosine,reinforce_only,1e-03,1e-05,1e-03,0.884,0.905,0.25,1e-05,384,20251216_2005_bs32_vocab10_repr384_msglen4_lr1...,20251217_1713_bs32_vocab10_repr384_msglen4_lr1...
1,1,True,celeba,cosine,reinforce_with_preservation,1e-03,1e-05,1e-04,0.897,0.918,0.25,1e-05,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251210_0653_bs32_vocab10_repr384_msglen4_lr1...
4,2,True,celeba,cosine,reinforce_with_preservation,1e-03,1e-06,1e-04,0.884,0.906,0.25,1e-05,384,20251216_2005_bs32_vocab10_repr384_msglen4_lr1...,20251217_2358_bs32_vocab10_repr384_msglen4_lr1...


In [412]:
mean_and_std(final)

,VQEL,dataset,sim,agent_a_training_mode,mutual_play_accuracy
0,True,celeba,cosine,frozen,90.4 ± 0.8
1,True,celeba,cosine,reinforce_only,91.3 ± 0.7
2,True,celeba,cosine,reinforce_with_preservation,90.6 ± nan


### VQ-Reinforce

In [413]:
add_heading(3, "VQ-Reinforce")

res = filter_df({
    "dataset": "celeba",
    "VQEL": True,
    "num_pretrain_epochs": 0
})

to_html(res[vq_rf_cols_report])

res[vq_rf_cols]

,seed,VQEL,dataset,learning_rate_phase2_a,learning_rate_phase2_b,mutual_play_accuracy,commitment_weight,sampling_temperature,representation_dim,agent_a_training_mode,num_pretrain_epochs,contrastive_loss_temperature,path
440,1,True,celeba,1e-03,1e-03,0.393,0.25,1e-01,384,reinforce_only,0,1e-02,20251206_1324_bs32_vocab10_repr384_msglen4_lr1...
350,1,True,celeba,1e-04,1e-04,0.903,0.25,1e-01,384,reinforce_only,0,1e-02,20251206_2352_bs32_vocab10_repr384_msglen4_lr1...
24,1,True,celeba,1e-05,1e-05,0.832,0.25,1e-01,384,reinforce_only,0,1e-02,20251207_0115_bs32_vocab10_repr384_msglen4_lr1...


### Baseline

In [414]:
add_heading(3, "Baseline")

res = filter_df({
    "dataset": "celeba",
    "VQEL": False,
}).sort_values(by=["contrastive_loss_temperature", "entropy_regularization_factor", "learning_rate"])

to_html(res[bs_cols_report])

res[bs_cols]

,VQEL,dataset,contrastive_loss_temperature,entropy_regularization_factor,learning_rate,test_accuracy,sampling_temperature,representation_dim,path
54,False,celeba,1e-03,1e-03,1e-03,0.34,1,384,20251130_1207_bs32_vocab10_repr384_msglen4_lr0...
229,False,celeba,1e-03,1e-03,1e-04,0.9,1,384,20251201_0129_bs32_vocab10_repr384_msglen4_lr0...
443,False,celeba,1e-03,1e-03,1e-05,0.742,1,384,20251201_0958_bs32_vocab10_repr384_msglen4_lr1...


---

# EXP2: $Self_B$ + $Mutual_{A, B, reinforce}$

In [415]:
line()
add_heading(1, "EXP2: Self(B) then Mutual(A, B, reinforce)")

write(
"""
Here, we train agent B in the self-play phase for 50 epochs, followed by another 50 epochs in mutual play.
During the mutual-play phase, agent B may be either frozen or unfrozen.
"""
)

In [416]:
vq_cols = [
    "seed",
    "VQEL",
    "dataset",
    "sim",
    "agent_a_training_mode",
    "freeze_agent_b", 
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_b",
    "mutual_play_accuracy",
    "commitment_weight",
    "sampling_temperature",
    "decay",
    "representation_dim",
    "pretrained_checkpoint_b",
    "path",
]

vq_cols_report = [
    "agent_a_training_mode",
    "freeze_agent_b",
    "learning_rate_phase1",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_b",
    "mutual_play_accuracy",
]

## Objects

In [417]:
add_heading(2, "Objects")

res = filter_df({
    "dataset": "objects",
    "sim": "cosine",
    "VQEL": True,
    "pretrained_checkpoint_a": "None",
    "pretrained_checkpoint_b": "!None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,freeze_agent_b,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_b,mutual_play_accuracy,commitment_weight,sampling_temperature,decay,representation_dim,pretrained_checkpoint_b,path
342,1,True,objects,cosine,reinforce_only,False,1e-04,1e-03,1e-03,0.826,0.03,0.25,1e-01,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_0544_bs32_vocab10_repr32_msglen4_lr1_...
274,1,True,objects,cosine,reinforce_only,False,1e-04,1e-03,1e-04,0.826,0.015,0.25,1e-01,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_0553_bs32_vocab10_repr32_msglen4_lr1_...
48,1,True,objects,cosine,reinforce_only,False,1e-04,1e-04,1e-04,0.826,0.02,0.25,1e-01,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_1835_bs32_vocab10_repr32_msglen4_lr1_...
348,1,True,objects,cosine,reinforce_only,False,1e-04,1e-03,1e-05,0.826,0.015,0.25,1e-01,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_0603_bs32_vocab10_repr32_msglen4_lr1_...
259,1,True,objects,cosine,reinforce_only,False,1e-04,1e-03,1e-06,0.826,0.019,0.25,1e-01,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_0612_bs32_vocab10_repr32_msglen4_lr1_...
337,1,True,objects,cosine,reinforce_only,False,1e-04,1e-03,1e-03,0.826,0.147,0.25,1e-02,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_1335_bs32_vocab10_repr32_msglen4_lr1_...
79,1,True,objects,cosine,reinforce_only,False,1e-04,1e-03,1e-04,0.826,0.013,0.25,1e-02,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_1344_bs32_vocab10_repr32_msglen4_lr1_...
363,1,True,objects,cosine,reinforce_only,False,1e-04,1e-04,1e-04,0.826,0.405,0.25,1e-02,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_1746_bs32_vocab10_repr32_msglen4_lr1_...
7,1,True,objects,cosine,reinforce_only,False,1e-04,1e-03,1e-05,0.826,0.014,0.25,1e-02,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_1354_bs32_vocab10_repr32_msglen4_lr1_...
127,1,True,objects,cosine,reinforce_only,False,1e-04,1e-04,1e-05,0.826,0.029,0.25,1e-02,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251214_1756_bs32_vocab10_repr32_msglen4_lr1_...


## ShapeWorld

In [418]:
add_heading(2, "ShapeWorld")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "pretrained_checkpoint_a": "None",
    "pretrained_checkpoint_b": "!None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,freeze_agent_b,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_b,mutual_play_accuracy,commitment_weight,sampling_temperature,decay,representation_dim,pretrained_checkpoint_b,path
328,1,True,shape,cosine,reinforce_only,False,1e-04,1e-04,1e-04,0.869,0.846,0.25,1e-01,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251214_0347_bs32_vocab10_repr1024_msglen4_lr...
400,1,True,shape,cosine,reinforce_only,False,1e-04,1e-04,1e-05,0.869,0.843,0.25,1e-01,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251214_0409_bs32_vocab10_repr1024_msglen4_lr...
89,1,True,shape,cosine,reinforce_only,False,1e-04,1e-05,1e-05,0.869,0.822,0.25,1e-01,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251214_0454_bs32_vocab10_repr1024_msglen4_lr...
160,1,True,shape,cosine,reinforce_only,False,1e-04,1e-04,1e-06,0.869,0.782,0.25,1e-01,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251214_0431_bs32_vocab10_repr1024_msglen4_lr...
228,1,True,shape,cosine,reinforce_only,True,1e-04,1e-03,1e-04,0.869,0.013,0.25,1e-01,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251216_1332_bs32_vocab10_repr1024_msglen4_lr...
422,1,True,shape,cosine,reinforce_only,True,1e-04,1e-04,1e-04,0.869,0.074,0.25,1e-01,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251216_1347_bs32_vocab10_repr1024_msglen4_lr...
288,1,True,shape,cosine,reinforce_only,True,1e-04,1e-05,1e-04,0.869,0.237,0.25,1e-01,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251216_1402_bs32_vocab10_repr1024_msglen4_lr...


## DSprites

In [419]:
add_heading(2, "DSprites")

res = filter_df({
    "dataset": "dsprites",
    "sim": "cosine",
    "VQEL": True,
    "pretrained_checkpoint_a": "None",
    "pretrained_checkpoint_b": "!None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,freeze_agent_b,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_b,mutual_play_accuracy,commitment_weight,sampling_temperature,decay,representation_dim,pretrained_checkpoint_b,path
249,1,True,dsprites,cosine,reinforce_only,False,1e-04,1e-04,1e-04,0.929,0.827,0.25,1e-01,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251214_0114_bs32_vocab10_repr1024_msglen4_lr...
290,1,True,dsprites,cosine,reinforce_only,False,1e-04,1e-04,1e-05,0.929,0.865,0.25,1e-01,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251214_0136_bs32_vocab10_repr1024_msglen4_lr...
267,1,True,dsprites,cosine,reinforce_only,False,1e-04,1e-05,1e-05,0.929,0.831,0.25,1e-01,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251214_0220_bs32_vocab10_repr1024_msglen4_lr...
319,1,True,dsprites,cosine,reinforce_only,False,1e-04,1e-04,1e-06,0.929,0.054,0.25,1e-01,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251214_0158_bs32_vocab10_repr1024_msglen4_lr...
29,1,True,dsprites,cosine,reinforce_only,True,1e-04,1e-03,1e-04,0.929,1e-02,0.25,1e-01,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251216_1247_bs32_vocab10_repr1024_msglen4_lr...
19,1,True,dsprites,cosine,reinforce_only,True,1e-04,1e-04,1e-04,0.929,0.031,0.25,1e-01,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251216_1302_bs32_vocab10_repr1024_msglen4_lr...
196,1,True,dsprites,cosine,reinforce_only,True,1e-04,1e-05,1e-04,0.929,0.341,0.25,1e-01,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251216_1316_bs32_vocab10_repr1024_msglen4_lr...


## CelebA

In [420]:
add_heading(2, "CelebA")

res = filter_df({
    "dataset": "celeba",
    "sim": "cosine",
    "VQEL": True,
    "pretrained_checkpoint_a": "None",
    "pretrained_checkpoint_b": "!None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,seed,VQEL,dataset,sim,agent_a_training_mode,freeze_agent_b,learning_rate_phase1,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_b,mutual_play_accuracy,commitment_weight,sampling_temperature,decay,representation_dim,pretrained_checkpoint_b,path
404,1,True,celeba,cosine,reinforce_only,False,1e-04,1e-03,1e-03,0.897,0.519,0.25,1e-01,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_0834_bs32_vocab10_repr384_msglen4_lr1...
16,1,True,celeba,cosine,reinforce_only,False,1e-04,1e-03,1e-04,0.897,0.041,0.25,1e-01,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_0920_bs32_vocab10_repr384_msglen4_lr1...
147,1,True,celeba,cosine,reinforce_only,False,1e-04,1e-03,1e-05,0.897,0.012,0.25,1e-01,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_1004_bs32_vocab10_repr384_msglen4_lr1...
441,1,True,celeba,cosine,reinforce_only,False,1e-04,1e-03,1e-06,0.897,0.014,0.25,1e-01,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_1049_bs32_vocab10_repr384_msglen4_lr1...
431,1,True,celeba,cosine,reinforce_only,True,1e-04,1e-03,1e-04,0.897,1e-02,0.25,1e-01,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251216_1417_bs32_vocab10_repr384_msglen4_lr1...
345,1,True,celeba,cosine,reinforce_only,True,1e-04,1e-04,1e-04,0.897,0.313,0.25,1e-01,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251216_1501_bs32_vocab10_repr384_msglen4_lr1...
403,1,True,celeba,cosine,reinforce_only,True,1e-04,1e-05,1e-04,0.897,0.446,0.25,1e-01,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251216_1527_bs32_vocab10_repr384_msglen4_lr1...


---

# EXP3: ($Self_A$ and $Self_B$) + $Mutual_{A, B, reinforce}$

In [421]:
line()
add_heading(1, "EXP3: Self(A) and Self(B) then Mutual(A, B, reinforce)")

write(
"""
Here, we first train agents A and B in a self-play phase with different seeds. 
Afterwards, they engage in mutual play with each other, during which their object encoders are frozen, 
and the agents can be fine-tuned using the REINFORCE algorithm.

To achieve better results, we investigate different configurations:
<b>- reset_unfrozen_params</b>: resets all unfrozen parameters (i.e. all network parameters except the object encoders)
<b>- freeze_codebook</b>: whether to freeze the codebook during mutual play
"""
)

In [422]:
vq_cols = [
    # "VQEL",
    "dataset",
    # "sim",
    "freeze_object_encoder",
    "reset_unfrozen_params",
    "freeze_codebook",
    "sampling_temperature",
    "commitment_weight",
    "entropy_regularization_factor",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "self_play_accuracy_b",
    "mutual_play_accuracy",
    "decay",
    "representation_dim",
    "pretrained_checkpoint_a",
    "pretrained_checkpoint_b",
    "path",
]

vq_cols_report = [
    "freeze_object_encoder",
    "reset_unfrozen_params",
    "freeze_codebook",
    "sampling_temperature",
    "commitment_weight",
    "entropy_regularization_factor",
    "learning_rate_phase2_a",
    "learning_rate_phase2_b",
    "self_play_accuracy_a",
    "self_play_accuracy_b",
    "mutual_play_accuracy"
]

## Objects

In [423]:
add_heading(2, "Objects")

res = filter_df({
    "dataset": "objects",
    "sim": "cosine",
    "VQEL": True,
    "pretrained_checkpoint_b": "!None",
    "pretrained_checkpoint_a": "!None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,dataset,freeze_object_encoder,reset_unfrozen_params,freeze_codebook,sampling_temperature,commitment_weight,entropy_regularization_factor,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,self_play_accuracy_b,mutual_play_accuracy,decay,representation_dim,pretrained_checkpoint_a,pretrained_checkpoint_b,path
112,objects,no,no,no,1e-01,0.25,0,1e-03,1e-03,0.826,0.814,0.88,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251211_1542_bs32_vocab10_repr32_msglen4_lr1_...
8,objects,no,no,no,1e-01,0.25,0,1e-04,1e-03,0.826,0.814,0.866,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251212_0226_bs32_vocab10_repr32_msglen4_lr1_...
157,objects,no,no,no,1e-01,0.25,0,1e-05,1e-03,0.826,0.814,0.768,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251212_0235_bs32_vocab10_repr32_msglen4_lr1_...
59,objects,no,no,no,1e-01,0.25,0,1e-06,1e-03,0.826,0.814,0.717,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251212_0244_bs32_vocab10_repr32_msglen4_lr1_...
67,objects,no,no,no,1e-01,0.25,0,1e-03,1e-04,0.826,0.814,0.858,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251212_0253_bs32_vocab10_repr32_msglen4_lr1_...
402,objects,no,no,no,1e-01,0.25,0,1e-04,1e-04,0.826,0.814,0.859,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251211_1551_bs32_vocab10_repr32_msglen4_lr1_...
77,objects,no,no,no,1e-01,0.25,0,1e-03,1e-05,0.826,0.814,0.439,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251212_0303_bs32_vocab10_repr32_msglen4_lr1_...
18,objects,no,no,no,1e-01,0.25,0,1e-05,1e-05,0.826,0.814,0.019,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251211_1600_bs32_vocab10_repr32_msglen4_lr1_...
151,objects,no,no,no,1e-01,0.25,0,1e-03,1e-06,0.826,0.814,0.038,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251212_0312_bs32_vocab10_repr32_msglen4_lr1_...
376,objects,no,no,no,1e-01,0.25,0,1e-06,1e-06,0.826,0.814,0.011,0.99,32,20251129_2130_bs32_vocab10_repr32_msglen4_lr1_...,20251207_1240_bs32_vocab10_repr32_msglen4_lr1_...,20251211_1610_bs32_vocab10_repr32_msglen4_lr1_...


## Shape

In [424]:
add_heading(2, "ShapeWorld")

res = filter_df({
    "dataset": "shape",
    "sim": "cosine",
    "VQEL": True,
    "pretrained_checkpoint_b": "!None",
    "pretrained_checkpoint_a": "!None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,dataset,freeze_object_encoder,reset_unfrozen_params,freeze_codebook,sampling_temperature,commitment_weight,entropy_regularization_factor,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,self_play_accuracy_b,mutual_play_accuracy,decay,representation_dim,pretrained_checkpoint_a,pretrained_checkpoint_b,path
32,shape,no,no,no,1e-01,0.25,0,1e-03,1e-03,0.869,0.888,0.713,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_1244_bs32_vocab10_repr1024_msglen4_lr...
150,shape,no,no,no,1e-01,0.25,0,1e-04,1e-04,0.869,0.888,0.854,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_1306_bs32_vocab10_repr1024_msglen4_lr...
330,shape,no,no,no,1e-01,0.25,0,1e-05,1e-04,0.869,0.888,0.912,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_2153_bs32_vocab10_repr1024_msglen4_lr...
339,shape,no,no,no,1e-01,0.25,0,1e-06,1e-04,0.869,0.888,0.729,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_2217_bs32_vocab10_repr1024_msglen4_lr...
305,shape,no,no,no,1e-01,0.25,0,1e-04,1e-05,0.869,0.888,0.859,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_2302_bs32_vocab10_repr1024_msglen4_lr...
379,shape,no,no,no,1e-01,0.25,0,1e-05,1e-05,0.869,0.888,0.875,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_1327_bs32_vocab10_repr1024_msglen4_lr...
387,shape,no,no,no,1e-01,0.25,0,1e-06,1e-05,0.869,0.888,0.65,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_2239_bs32_vocab10_repr1024_msglen4_lr...
163,shape,no,no,no,1e-01,0.25,0,1e-04,1e-06,0.869,0.888,0.773,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_2325_bs32_vocab10_repr1024_msglen4_lr...
80,shape,no,no,no,1e-01,0.25,0,1e-05,1e-06,0.869,0.888,0.72,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_2348_bs32_vocab10_repr1024_msglen4_lr...
415,shape,no,no,no,1e-01,0.25,0,1e-06,1e-06,0.869,0.888,0.062,0.99,1024,20251125_0053_bs32_vocab10_repr1024_msglen4_lr...,20251207_1320_bs32_vocab10_repr1024_msglen4_lr...,20251211_1350_bs32_vocab10_repr1024_msglen4_lr...


## DSprites

In [425]:
add_heading(2, "DSprites")

res = filter_df({
    "dataset": "dsprites",
    "sim": "cosine",
    "VQEL": True,
    "pretrained_checkpoint_b": "!None",
    "pretrained_checkpoint_a": "!None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,dataset,freeze_object_encoder,reset_unfrozen_params,freeze_codebook,sampling_temperature,commitment_weight,entropy_regularization_factor,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,self_play_accuracy_b,mutual_play_accuracy,decay,representation_dim,pretrained_checkpoint_a,pretrained_checkpoint_b,path
46,dsprites,no,no,no,1e-01,0.25,0,1e-03,1e-03,0.929,0.889,0.731,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251211_1412_bs32_vocab10_repr1024_msglen4_lr...
103,dsprites,no,no,no,1e-01,0.25,0,1e-04,1e-04,0.929,0.889,0.873,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251211_1434_bs32_vocab10_repr1024_msglen4_lr...
47,dsprites,no,no,no,1e-01,0.25,0,1e-05,1e-04,0.929,0.889,0.921,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251212_0011_bs32_vocab10_repr1024_msglen4_lr...
49,dsprites,no,no,no,1e-01,0.25,0,1e-06,1e-04,0.929,0.889,0.915,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251212_0033_bs32_vocab10_repr1024_msglen4_lr...
220,dsprites,no,no,no,1e-01,0.25,0,1e-04,1e-05,0.929,0.889,0.867,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251212_0118_bs32_vocab10_repr1024_msglen4_lr...
397,dsprites,no,no,no,1e-01,0.25,0,1e-05,1e-05,0.929,0.889,0.931,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251211_1457_bs32_vocab10_repr1024_msglen4_lr...
68,dsprites,no,no,no,1e-01,0.25,0,1e-06,1e-05,0.929,0.889,0.917,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251212_0055_bs32_vocab10_repr1024_msglen4_lr...
226,dsprites,no,no,no,1e-01,0.25,0,1e-04,1e-06,0.929,0.889,0.81,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251212_0140_bs32_vocab10_repr1024_msglen4_lr...
91,dsprites,no,no,no,1e-01,0.25,0,1e-05,1e-06,0.929,0.889,0.87,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251212_0203_bs32_vocab10_repr1024_msglen4_lr...
360,dsprites,no,no,no,1e-01,0.25,0,1e-06,1e-06,0.929,0.889,0.864,0.99,1024,20251124_2131_bs32_vocab10_repr1024_msglen4_lr...,20251207_1532_bs32_vocab10_repr1024_msglen4_lr...,20251211_1519_bs32_vocab10_repr1024_msglen4_lr...


## CelebA

In [426]:
add_heading(2, "CelebA")

res = filter_df({
    "dataset": "celeba",
    "sim": "cosine",
    "VQEL": True,
    "pretrained_checkpoint_b": "!None",
    "pretrained_checkpoint_a": "!None",
})

to_html(res[vq_cols_report])

res[vq_cols]

,dataset,freeze_object_encoder,reset_unfrozen_params,freeze_codebook,sampling_temperature,commitment_weight,entropy_regularization_factor,learning_rate_phase2_a,learning_rate_phase2_b,self_play_accuracy_a,self_play_accuracy_b,mutual_play_accuracy,decay,representation_dim,pretrained_checkpoint_a,pretrained_checkpoint_b,path
116,celeba,no,no,no,1e-01,0.25,0,1e-03,1e-03,0.897,0.89,0.894,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_2233_bs32_vocab10_repr384_msglen4_lr1...,20251215_0203_bs32_vocab10_repr384_msglen4_lr1...
390,celeba,no,no,no,1e-01,0.25,0,1e-04,1e-03,0.897,0.89,0.934,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_2233_bs32_vocab10_repr384_msglen4_lr1...,20251215_0250_bs32_vocab10_repr384_msglen4_lr1...
237,celeba,no,no,no,1e-01,0.25,0,1e-05,1e-03,0.897,0.89,0.907,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_2233_bs32_vocab10_repr384_msglen4_lr1...,20251215_0336_bs32_vocab10_repr384_msglen4_lr1...
175,celeba,no,no,no,1e-01,0.25,0,1e-06,1e-03,0.897,0.89,0.884,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_2233_bs32_vocab10_repr384_msglen4_lr1...,20251215_0421_bs32_vocab10_repr384_msglen4_lr1...
45,celeba,no,no,no,1e-01,0.25,0,1e-04,1e-04,0.897,0.89,0.934,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_2233_bs32_vocab10_repr384_msglen4_lr1...,20251214_2344_bs32_vocab10_repr384_msglen4_lr1...
186,celeba,no,no,no,1e-01,0.25,0,1e-05,1e-04,0.897,0.89,0.927,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_2233_bs32_vocab10_repr384_msglen4_lr1...,20251215_0030_bs32_vocab10_repr384_msglen4_lr1...
144,celeba,no,no,no,1e-01,0.25,0,1e-06,1e-04,0.897,0.89,0.894,0.99,384,20251201_1456_bs32_vocab10_repr384_msglen4_lr1...,20251214_2233_bs32_vocab10_repr384_msglen4_lr1...,20251215_0116_bs32_vocab10_repr384_msglen4_lr1...


---